In [3]:
import json
import pandas as pd
from datetime import datetime
import re

# Read JSON
with open("data.json", "r", encoding="utf-8") as f:
    document_dict = json.load(f)
data = document_dict

records = []
for year in ["promet_2022", "promet_2023", "promet_2024"]:
    for month in document_dict[year]["months_data"].keys():
        month_data = document_dict[year]["months_data"][month]
        file_data_dict = month_data["file_data"]

        # Create a list of flattened records
        for file_name, file_details in file_data_dict.items():
            record = {
                # File-specific fields
                "year": year,
                "month": month,
                "file_name": file_name,
                "content": file_details["content"],
                "file_word_count": file_details["word_count"],
                "file_char_count": file_details["char_count"],
                "file_sentence_count": file_details["sentence_count"],
                "sentences": file_details["sentences"],
                
                # Month-specific fields
                "month_files": month_data["files"],
                "month_sentence_count": month_data["sentence_count"],
                "month_word_count": month_data["word_count"],

                # Year-specific fields
                "months": document_dict[year]["months"],
                "year_sentence_count": document_dict[year]["sentence_count"],
                "year_word_count": document_dict[year]["word_count"]
            }

            # Add datetime parsing
            content_text = file_details["content"]
            match = re.search(r"Prometne informacije\s+(\d{1,2}\.\s*\d{1,2}\.\s*\d{4})\s+(\d{1,2}\.\d{2})", content_text)
            if match:
                date_part = match.group(1).replace(" ", "") 
                time_part = match.group(2)                
                datetime_str = f"{date_part} {time_part}"
                try:
                    parsed_datetime = datetime.strptime(datetime_str, "%d.%m.%Y %H.%M")
                except ValueError:
                    parsed_datetime = None
            else:
                parsed_datetime = None
            record["datetime"] = parsed_datetime

            records.append(record)

# Convert to DataFrame
RTFS = pd.DataFrame(records)
RTFS.head()

,year,month,file_name,content,file_word_count,file_char_count,file_sentence_count,sentences,month_files,month_sentence_count,month_word_count,months,year_sentence_count,year_word_count,datetime
0,promet_2022,april_2022,1.rtf,Prometne informacije 23. 04. 2022 14.30 2. pro...,62,398,4,[Prometne informacije 23. 04. 2022 14.30 2. pr...,741,3832,55464,12,51490,740176,2022-04-23 14:30:00
1,promet_2022,april_2022,10.rtf,Prometne informacije 23. 04. 2022 13.00 1. in ...,70,437,5,[Prometne informacije 23. 04. 2022 13.00 1. in...,741,3832,55464,12,51490,740176,2022-04-23 13:00:00
2,promet_2022,april_2022,100.rtf,Prometne informacije 08. 04. 2022 16.30 2. pro...,109,739,7,[Prometne informacije 08. 04. 2022 16.30 2. pr...,741,3832,55464,12,51490,740176,2022-04-08 16:30:00
3,promet_2022,april_2022,101.rtf,Prometne informacije 17.04. 2022 07.00 1. in 2...,68,417,6,"[Prometne informacije 17.04., 2022 07.00 1. in...",741,3832,55464,12,51490,740176,2022-04-17 07:00:00
4,promet_2022,april_2022,102.rtf,Prometne informacije 17.04. 2022 06.30 1. prog...,57,329,5,"[Prometne informacije 17.04., 2022 06.30 1. pr...",741,3832,55464,12,51490,740176,2022-04-17 06:30:00


In [4]:
# Saving RTFS to CSV

RTFS_reduced = RTFS[["file_name", "content", "datetime"]]
RTFS_reduced["file_name"] = range(1, len(RTFS) + 1) # Unique file names
RTFS_reduced.to_csv("rtfs_reduced.csv", index=False, encoding='utf-8')

/tmp/ipykernel_6994/1067068591.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RTFS_reduced["file_name"] = range(1, len(RTFS) + 1) # Unique file names


In [6]:
# Read the "traffic_reports_2022_2023_2024.xlsx" to Pandas DataFrame

path = "../Data/traffic_reports_2022_2023_2024.xlsx"
years = [2022, 2023, 2024]
REPORTS = {}

def clean_cell(val):
    if isinstance(val, str):
        val = re.sub(r'<strong>.*?</strong>', '', val, flags=re.DOTALL)  # removes <strong>...</strong>
        val = re.sub(r"<.*?>", "", val) # removes all HTML tags
        val = re.sub(r'"', '', val) # removes double quotes
        val = val.strip() # removes spaces             
    return val

for year in years:
    REPORTS[year] = pd.read_excel(path, sheet_name=str(year))

    "Some data cleaning and preprocessing"

    # Drop 100% empty columns A2 and C1
    REPORTS[year].drop(columns=["A2", "C1"], inplace=True)

    # Drop English columns
    REPORTS[year].drop(columns=["B2", "C2"], inplace=True)

    # Drop useless columns (report's id, report's author, all titles)
    REPORTS[year].drop(columns=["LegacyId", "Operater", "TitlePomembnoSLO", "TitleNesreceSLO", "TitleZastojiSLO", "TitleVremeSLO", "TitleOvireSLO","TitleDeloNaCestiSLO", "TitleOpozorilaSLO", "TitleMednarodneInformacijeSLO", "TitleSplosnoSLO"], inplace=True)

    # Clean up text values
    REPORTS[year] = REPORTS[year].applymap(clean_cell)

REPORTS[2022].head(20)

/tmp/ipykernel_6994/3380724407.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  REPORTS[year] = REPORTS[year].applymap(clean_cell)
/tmp/ipykernel_6994/3380724407.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  REPORTS[year] = REPORTS[year].applymap(clean_cell)
/tmp/ipykernel_6994/3380724407.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  REPORTS[year] = REPORTS[year].applymap(clean_cell)


,Datum,A1,B1,ContentPomembnoSLO,ContentNesreceSLO,ContentZastojiSLO,ContentVremeSLO,ContentOvireSLO,ContentDeloNaCestiSLO,ContentOpozorilaSLO,ContentMednarodneInformacijeSLO,ContentSplosnoSLO
0,2022-01-01 00:07:07,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
1,2022-01-01 00:07:29,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
2,2022-01-01 00:07:30,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
3,2022-01-01 00:07:36,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
4,2022-01-01 00:16:26,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
5,2022-01-01 00:21:22,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
6,2022-01-01 02:23:03,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
7,2022-01-01 04:09:46,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
8,2022-01-01 04:33:31,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
9,2022-01-01 04:45:26,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN


In [10]:
# For each report, find corresponding RTF file. This takes about 2 minutes to run.
RTFS_reduced = RTFS_reduced.sort_values("datetime").reset_index(drop=True)
RTFS_reduced['datetime'] = pd.to_datetime(RTFS_reduced['datetime'])

for year in years:
    REPORTS[year]["RTF_file_name"] = None
    
    for index, row in REPORTS[year].iterrows():
        report_date = datetime.strptime(str(row["Datum"]), "%Y-%m-%d %H:%M:%S") #- timedelta(hours=1)
        # Find first RTF document with its datetime after the current report's datetime
        match = RTFS_reduced[RTFS_reduced["datetime"] >= report_date]

        if not match.empty:
            matched_file = match.iloc[0]["file_name"]    #  # Get the first match

        else:
            matched_file = None  #  # Get the first match

        REPORTS[year].at[index, "RTF_file_name"] = matched_file

In [11]:
# Remove report data without corresponding RTF file name
for year in years:
    REPORTS[year] = REPORTS[year][REPORTS[year]["RTF_file_name"].notna()].reset_index(drop=True)

In [12]:
# Save new reports data to one CSV file
all_reports = pd.concat([REPORTS[year] for year in years], ignore_index=True)
all_reports.to_csv("traffic_reports_linked.csv", index=False, encoding='utf-8')

In [14]:
REPORTS = pd.read_csv("traffic_reports_linked.csv", encoding='utf-8')
RTFS = pd.read_csv("rtfs_reduced.csv", encoding='utf-8')

In [15]:
from sklearn.model_selection import train_test_split

RTFS_train, temp = train_test_split(RTFS, test_size=0.3, random_state=42)
RTFS_valid, RTFS_test = train_test_split(temp, test_size=0.5, random_state=42)
print(f"RTFS size: {len(RTFS)}")
print(f"RTFS_train size: {len(RTFS_train)}")
print(f"RTFS_valid size: {len(RTFS_valid)}")
print(f"RTFS_test size: {len(RTFS_test)}")

RTFS size: 28037
RTFS_train size: 19625
RTFS_valid size: 4206
RTFS_test size: 4206


In [16]:
def prepare_input(RTF_file_name):

    # Traffic data for the given RTF file name
    reports = REPORTS[REPORTS['RTF_file_name'] == RTF_file_name]

    # Input data used for generating desired LLM output
    input = {col: set(reports[col].dropna()) for col in [
        'Datum',
        'A1', 
        'B1', 
        'ContentPomembnoSLO', 
        'ContentNesreceSLO', 
        'ContentZastojiSLO', 
        'ContentVremeSLO', 
        'ContentOvireSLO', 
        'ContentDeloNaCestiSLO', 
        'ContentOpozorilaSLO',
        'ContentMednarodneInformacijeSLO', 
        'ContentSplosnoSLO']}
    
    lines = ["Vhodni podatki:"]
    
    if input['ContentPomembnoSLO']:
        lines.append(f"- Zelo pomembne informacije o prometu: {', '.join(map(str, input['ContentPomembnoSLO']))}")
    if input['A1']:
        lines.append(f"- Pomembne informacije o prometu: {', '.join(map(str, input['A1']))}")
    if input['B1']:
        lines.append(f"- Manj pomembne informacije o prometu: {', '.join(map(str, input['B1']))}")
    if input['ContentNesreceSLO']:
        lines.append(f"- Informacije o nesrečah: {', '.join(map(str, input['ContentNesreceSLO']))}")
    if input['ContentZastojiSLO']:
        lines.append(f"- Informacije o zastojih: {', '.join(map(str, input['ContentZastojiSLO']))}")
    if input['ContentVremeSLO']:
        lines.append(f"- Informacije o vremenu: {', '.join(map(str, input['ContentVremeSLO']))}")
    if input['ContentOvireSLO']:
        lines.append(f"- Informacije o ovirah: {', '.join(map(str, input['ContentOvireSLO']))}")
    if input['ContentDeloNaCestiSLO']:
        lines.append(f"- Informacije o delu na cesti: {', '.join(map(str, input['ContentDeloNaCestiSLO']))}")
    if input['ContentOpozorilaSLO']:
        lines.append(f"- Informacije o opozorilih: {', '.join(map(str, input['ContentOpozorilaSLO']))}")
    if input['ContentMednarodneInformacijeSLO']:
        lines.append(f"- Informacije o mednarodnih informacijah: {', '.join(map(str, input['ContentMednarodneInformacijeSLO']))}")
    if input['ContentSplosnoSLO']:
        lines.append(f"- Splošne informacije: {', '.join(map(str, input['ContentSplosnoSLO']))}")

    return lines

def generate_shot(RTFS_, RTF_file_name):
    
    # Preparing example shot text
    shot = prepare_input(RTF_file_name)

    # Desired LLM output
    output = RTFS_[RTFS_['file_name'] == RTF_file_name]["content"].values[0]

    shot.append("")
    shot.append(f"Poročilo: {output}\n")

    return '\n'.join(shot)

In [17]:
# Generate the prompts. This takes about a minute to run.
RTFS_train["prompt"] = None
RTFS_test["prompt"] = None
RTFS_valid["prompt"] = None
for index, row in RTFS_train.iterrows():
    shot = generate_shot(RTFS_train, row['file_name'])
    if shot:
        RTFS_train.at[index, 'prompt'] =  shot

for index, row in RTFS_test.iterrows():
    shot = generate_shot(RTFS_test, row['file_name'])
    if shot:
        RTFS_test.at[index, 'prompt'] =  shot

for index, row in RTFS_valid.iterrows():
    shot = generate_shot(RTFS_valid, row['file_name'])
    if shot:
        RTFS_valid.at[index, 'prompt'] =  shot


In [18]:
train_dataset = RTFS_train[RTFS_train["prompt"].notnull()]
test_dataset = RTFS_test[RTFS_test["prompt"].notnull()]
valid_dataset = RTFS_valid[RTFS_valid["prompt"].notnull()]

In [34]:
train_dataset.to_csv("train_dataset.csv", index=False, encoding='utf-8')
test_dataset.to_csv("test_dataset.csv", index=False, encoding='utf-8')
valid_dataset.to_csv("valid_dataset.csv", index=False, encoding='utf-8')

In [39]:
def convert_to_jsonl(dataset_name, expected_response_column="content"):
    dataset = pd.read_csv(dataset_name, encoding='utf-8')
    # Before we had the instructions and expected response in the prompt. Separate them now.
    dataset['prompt'] = dataset['prompt'].apply(lambda x: x.split("### Poročilo")[0].strip())
    
    # Also, add additional instructions to the prompt.
    instructions = \
"""
Generiraj poročilo o prometu na osnovi vhodnih podatkov, ki se začnejo z '### Vhodni podatki'. Odgovor naj vsebuje samo poročilo. Odgovarjaj v povedih.
Drži se hierarhije dogodkov (od najpomembnejših do najmanj pomembnih): 
- Voznik vozi v napačno smer  
- Zaprta avtocesta 
- Nesreča z zastojem na avtocesti 
- Zastoji zaradi del na avtocesti (ob krajših zastojih se pogosto dogajajo naleti) 
- Zaradi nesreče zaprta glavna ali regionalna cesta 
- Nesreče na avtocestah in drugih cestah 
- Pokvarjena vozila, ko je zaprt vsaj en prometni pas 
- Žival, ki je zašla na vozišče 
- Predmet/razsut tovor na avtocesti 
- Dela na avtocesti, kjer je večja nevarnost naleta (zaprt prometni pas, pred predori, v predorih, …) 
- Zastoj pred Karavankami in mejnimi prehodi 
Pomembno je sporočiti, če voznik ne vozi več v napačno smer ali če je konec zastojev zaradi katere koli prometne nesreče.
"""
    instructions = instructions.strip() + "\n\n"
    with open(dataset_name.replace(".csv", ".jsonl"), "w", encoding="utf-8") as f:
        for _, row in dataset.iterrows():
            prompt = instructions + row['prompt']
            expected_response = row[expected_response_column]
            json_line = {
                "prompt": prompt,
                "completion": expected_response
            }
            json.dump(json_line, f)

In [36]:
# These are the three datasets with the original reports.
convert_to_jsonl("train_dataset.csv")
convert_to_jsonl("test_dataset.csv")
convert_to_jsonl("valid_dataset.csv")

In [37]:
# We also generated the reports using gemini-2.0-flash (see generate_gemini_reports). The results are in "gemini_responses.txt".
import pandas as pd

train_dataset = pd.read_csv("train_dataset.csv")
train_dataset["generated_response"] = ""
with open("gemini_responses.txt", "r") as f:
    lines = f.readlines()
current_index = 0
current_response = []

for line in lines:
    if line.startswith("Index"):
        # The index written in the file has 10 added. Oops
        current_index = int(line.split(":")[0].split(" ")[1]) - 10
        current_response = []
    elif line.startswith("End of index"):
        reports = "\n".join(current_response).split("### Poročilo")

        # Sometimes the first report starts with ### Poročilo, sometimes it doesn't. Ignore the first one if it's empty. 
        if len(reports) == 11 and reports[0] == "":
            reports = reports[1:]

        if len(reports) == 11 and reports[0] != "":
            temp = 0

        if len(reports) != 10:
            continue

        for i in range(10):
            train_dataset.at[current_index + i, "generated_response"] = reports[i].strip()
    else:
        current_response.append(line)

num_no_response = sum(train_dataset["generated_response"] == "")
train_dataset_clean = train_dataset[train_dataset["generated_response"] != ""]

train_dataset.to_csv("train_dataset_generated_reports.csv", index=False)

In [40]:
# The dataset with generated reports
convert_to_jsonl("train_dataset_generated_reports.csv", expected_response_column="generated_response")